# 07 · Evaluación offline: SmolVLA contra la primera ACT

La misma vara que `05`: en los 7 episodios que ninguno de los dos vio al
entrenar, en distintos instantes, cada política recibe la observación grabada
y predice su bloque; lo comparamos con lo que hizo la persona. Las dos
referencias tontas de siempre dan la escala: quedarse quieto, y repetir la
última acción.

Ojo con lo que esto mide: el modelo siempre recibe la observación verdadera,
nunca la consecuencia de sus propias acciones. Dice si aprendió, no si la
tarea saldría en el robot.

In [ ]:
import torch  # antes que lerobot: evita un import circular entre torchvision y torch._dynamo
import json
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from lerobot.datasets.lerobot_dataset import LeRobotDataset

from toolkit.eval.offline import evaluate_episode
from toolkit.paths import CHECKPOINTS_DIR, DATASETS_DIR, REPO_ROOT
from toolkit.policy.load import LoadedPolicy, load_policy
from toolkit.policy.smolvla_g1_base import load_g1_policy

plt.rcParams["figure.dpi"] = 110
device = "cuda" if torch.cuda.is_available() else "cpu"

ds = LeRobotDataset("local/pick_mameluco_act_v1", root=DATASETS_DIR / "local" / "pick_mameluco_act_v1")
split = json.loads((REPO_ROOT / "notes" / "act-split.json").read_text())
names = ds.meta.features["action"]["names"]
JOINTS = names[0] if isinstance(names[0], list) else names

act = load_policy(CHECKPOINTS_DIR / "act_cheap_v1" / "pretrained_model", device=device)
train_meta = LeRobotDataset("local/pick_mameluco_act_v1_train", root=DATASETS_DIR / "local" / "pick_mameluco_act_v1_train").meta
smol_policy, smol_pre, smol_post = load_g1_policy(
    CHECKPOINTS_DIR / "smolvla_g1_v1" / "checkpoints" / "last" / "pretrained_model", train_meta.stats, device)
smol = LoadedPolicy(smol_policy, smol_pre, smol_post, [])
print("bloque · ACT:", act.policy.config.chunk_size, "· SmolVLA:", smol.policy.config.chunk_size)

Los bloques no tienen el mismo largo (ACT 48, SmolVLA 50). Para comparar
igual, evaluamos a los dos en los **mismos instantes** (cada 48 cuadros) y
medimos sólo las primeras 48 acciones de cada bloque.

In [ ]:
HORIZON = 48
torch.manual_seed(0)
predictions = {"ACT": [], "SmolVLA": []}
for episode in split["heldout_episode_indices"]:
    predictions["ACT"] += evaluate_episode(act, ds, episode, stride=HORIZON)
    predictions["SmolVLA"] += evaluate_episode(smol, ds, episode, stride=HORIZON)

common = {(p.episode_index, p.start) for p in predictions["ACT"]} & {(p.episode_index, p.start) for p in predictions["SmolVLA"]}
by_key = {name: {(p.episode_index, p.start): p for p in preds if (p.episode_index, p.start) in common}
          for name, preds in predictions.items()}
keys = sorted(common)
print(f"{len(keys)} bloques en {len(split['heldout_episode_indices'])} episodios no vistos")


def errors(name, field):
    """Error absoluto (rad) por bloque, paso y articulación: (bloques, 48, 26)."""
    return np.stack([np.abs(getattr(by_key[name][k], field)[:HORIZON] - by_key[name][k].target[:HORIZON]) for k in keys])


table = {
    "ACT": errors("ACT", "predicted"),
    "SmolVLA": errors("SmolVLA", "predicted"),
    "quedarse quieto": errors("ACT", "hold"),
    "repetir la última": errors("ACT", "last_action"),
}
summary = pd.DataFrame({name: {"error medio (rad)": e.mean(), "error medio (°)": np.degrees(e.mean())} for name, e in table.items()}).T
summary.round(4)

*(Interpretación pendiente: se escribe con el modelo final del entrenamiento.)*

## Según cuán lejos predice

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.8))
styles = {"ACT": ("tab:blue", "-"), "SmolVLA": ("tab:red", "-"), "quedarse quieto": ("gray", "--"), "repetir la última": ("gray", ":")}
for name, e in table.items():
    color, style = styles[name]
    ax.plot(np.arange(HORIZON) / 30, np.degrees(e.mean(axis=(0, 2))), color=color, ls=style, label=name)
ax.set_xlabel("segundos hacia adelante")
ax.set_ylabel("error medio (°)")
ax.legend()
plt.tight_layout()

*(Interpretación pendiente: se escribe con el modelo final del entrenamiento.)*

## Por articulación

In [ ]:
per_joint = pd.DataFrame({name: np.degrees(table[name].mean(axis=(0, 1))) for name in ["ACT", "SmolVLA", "quedarse quieto"]},
                         index=JOINTS).sort_values("SmolVLA")
ax = per_joint.plot.barh(figsize=(8, 8), color=["tab:blue", "tab:red", "lightgray"])
ax.set_xlabel("error medio (°)")
plt.tight_layout()
better = (per_joint["SmolVLA"] < per_joint["ACT"]).sum()
print(f"SmolVLA tiene menos error que ACT en {better} de 26 articulaciones")

*(Interpretación pendiente: se escribe con el modelo final del entrenamiento.)*

## Un episodio entero

El codo derecho en un episodio no visto: en gris lo que hizo la persona, y los
bloques de cada política desde cada instante.

In [ ]:
episode = split["heldout_episode_indices"][1]
joint = JOINTS.index("kRightElbow")
rows = ds.meta.episodes
start, stop = int(rows["dataset_from_index"][episode]), int(rows["dataset_to_index"][episode])
truth = np.stack(ds.hf_dataset.with_format("numpy")[start:stop]["action"])

fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True)
for ax, name, color in [(axes[0], "ACT", "tab:blue"), (axes[1], "SmolVLA", "tab:red")]:
    ax.plot(truth[:, joint], color="0.6", lw=3, label="la persona")
    for (e, s), p in by_key[name].items():
        if e == episode:
            ax.plot(range(s, s + HORIZON), p.predicted[:HORIZON, joint], color=color, lw=1.2)
    ax.set_title(f"{name} · episodio {episode} · {JOINTS[joint]}", fontsize=10)
axes[1].set_xlabel("cuadro")
axes[0].legend(fontsize=8)
plt.tight_layout()

## Cuánto tarda

En la misma máquina, un bloque de cada una:

In [ ]:
frame = ds[int(rows["dataset_from_index"][episode]) + 100]


def timed(loaded, repeats=10):
    from toolkit.eval.offline import predict_chunk
    predict_chunk(loaded, frame)
    if device == "cuda":
        torch.cuda.synchronize()
    start = time.perf_counter()
    for _ in range(repeats):
        predict_chunk(loaded, frame)
    if device == "cuda":
        torch.cuda.synchronize()
    return 1000 * (time.perf_counter() - start) / repeats


print(f"ACT: {timed(act):.0f} ms por bloque · SmolVLA: {timed(smol):.0f} ms por bloque ({device})")

*(Interpretación pendiente: se escribe con el modelo final del entrenamiento.)*

## Lo que nos llevamos

*(Interpretación pendiente: se escribe con el modelo final del entrenamiento.)*

## Para que la comparación sea justa (lo que no es igual)

- **Entrenamiento distinto.** La primera ACT se entrenó fuera de este repo (30.000
  pasos, con la fuga de normalización de `notes/act-first-model.md`). SmolVLA,
  20.000 pasos con lote 32 y estadísticas sólo de entrenamiento.
- **Un solo sorteo.** SmolVLA sortea; acá medimos un sorteo por instante.
- **Sin lazo cerrado.** Lo que importa al final es si agarra el mameluco en el
  robot, y eso esta evaluación no lo dice.